In [1]:
!pip install -q 'setuptools<81'


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [22]:
# import kagglehub


# kagglehub.login()

# # Download latest version
# path = kagglehub.competition_download('neural-debris-removal-in-streak-detection-models')

# print("Path to competition files:", path)

In [2]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

/home/kiryl/neural-debris/venv/lib/python3.12/site-packages/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# ── Input paths (Kaggle Datasets) ──
POISONED_WEIGHTS = "neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
TEST_DIR         = "neural-debris-removal-in-streak-detection-models/test_set/test_set"

# ── Output paths ──
SUBMISSION_PATH = "neural-debris-removal-in-streak-detection-models/submission.csv"

# ── Model architecture (MUST match the poisoned model's training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Inference ──
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

In [4]:
class UInt16DatasetMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 in [0, 255] and attaches empty instances (the unlearning signal)."""
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)

        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

In [5]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))

cfg.MODEL.WEIGHTS = POISONED_WEIGHTS
cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES

predictor = DefaultPredictor(cfg)


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])

    # Use a single space for empty predictions so Kaggle's null-check passes.
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

Loading config /home/kiryl/neural-debris/venv/lib/python3.12/site-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Running inference on 2000 images...


Inference:   0%|          | 0/2000 [00:00<?, ?it/s]/home/kiryl/neural-debris/venv/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4382.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Inference: 100%|██████████| 2000/2000 [02:10<00:00, 15.37it/s]


Wrote neural-debris-removal-in-streak-detection-models/submission.csv  (2000 rows)


,id,image_id,prediction_string
0,0,0,0.708322 893.96 186.20 9.06 40.97 0.602179 208...
1,1,1,0.207476 541.47 428.21 25.57 10.42
2,2,10,0.914567 4.98 17.87 37.53 58.97 0.375204 28.86...
3,3,100,0.491502 997.92 769.79 21.76 52.10 0.390374 58...
4,4,1000,0.252510 1008.90 582.88 8.93 39.36
